In [45]:
# !pip install -q transformers datasets torch scikit-learn


In [46]:
from datasets import load_dataset
from transformers import DistilBertTokenizerFast, TFDistilBertForSequenceClassification
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
import tensorflow as tf

In [47]:
FINAL_LABELS = ['happy', 'sad', 'angry', 'fear', 'surprise', 'neutral']


In [48]:
EMOTION_MAPPING = {
    "happy": ["joy", "amusement", "optimism", "excitement", "love", "pride", "relief"],
    "sad": ["sadness", "grief", "remorse", "disappointment", "embarrassment"],
    "angry": ["anger", "annoyance", "disapproval"],
    "fear": ["fear", "nervousness"],
    "surprise": ["surprise", "realization"],
    "neutral": ["neutral", "confusion", "curiosity"]
}

LABEL2ID = {label: idx for idx, label in enumerate(FINAL_LABELS)}
ID2LABEL = {idx: label for label, idx in LABEL2ID.items()}

REVERSE_MAP = {}
for k, v in EMOTION_MAPPING.items():
    for emo in v:
        REVERSE_MAP[emo] = k


In [49]:
REVERSE_MAP

{'joy': 'happy',
 'amusement': 'happy',
 'optimism': 'happy',
 'excitement': 'happy',
 'love': 'happy',
 'pride': 'happy',
 'relief': 'happy',
 'sadness': 'sad',
 'grief': 'sad',
 'remorse': 'sad',
 'disappointment': 'sad',
 'embarrassment': 'sad',
 'anger': 'angry',
 'annoyance': 'angry',
 'disapproval': 'angry',
 'fear': 'fear',
 'nervousness': 'fear',
 'surprise': 'surprise',
 'realization': 'surprise',
 'neutral': 'neutral',
 'confusion': 'neutral',
 'curiosity': 'neutral'}

In [ ]:
from datasets import load_dataset

dataset = load_dataset("go_emotions")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})


In [51]:
PRIORITY = [
    "anger", "fear", "sadness", "disgust",
    "surprise", "joy", "neutral"
]

# Extract the label converter function once before mapping
# Assuming the 'labels' feature structure is consistent across all splits.
label_int2str_mapper = dataset["train"].features["labels"].feature.int2str

def map_labels(example):
    # Convert integer labels from 'labels' column to their string representations
    original_label_names = [label_int2str_mapper(label_id) for label_id in example["labels"]]

    for emo in PRIORITY:
        if emo in original_label_names and emo in REVERSE_MAP:
            example["label"] = LABEL2ID[REVERSE_MAP[emo]]
            return example
    example["label"] = -1
    return example


dataset = dataset.map(map_labels)
dataset = dataset.filter(lambda x: x["label"] != -1)
# Select all available examples in the filtered train set, as 30000 was out of bounds
dataset["train"] = dataset["train"].shuffle(seed=42).select(range(len(dataset["train"])))

Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

Filter:   0%|          | 0/43410 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5426 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5427 [00:00<?, ? examples/s]

In [52]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize_data(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=160
    )

tokenized_dataset = dataset.map(tokenize_data, batched=True)


Map:   0%|          | 0/19896 [00:00<?, ? examples/s]

Map:   0%|          | 0/2469 [00:00<?, ? examples/s]

Map:   0%|          | 0/2487 [00:00<?, ? examples/s]

In [53]:
from transformers import TFDistilBertForSequenceClassification

model = TFDistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(FINAL_LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    from_pt=True
)


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_projector.weight', 'vocab_transform.bias', 'vocab_projector.bias', 'vocab_layer_norm.bias', 'vocab_transform.weight', 'vocab_layer_norm.weight']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'cla

In [54]:
from tf_keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss=SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [55]:
from transformers import DefaultDataCollator

data_collator = DefaultDataCollator(return_tensors="tf")

tf_train_dataset = model.prepare_tf_dataset(
    tokenized_dataset["train"],
    shuffle=True,
    batch_size=32,
    collate_fn=data_collator,
)

tf_val_dataset = model.prepare_tf_dataset(
    tokenized_dataset["validation"],
    shuffle=False,
    batch_size=32,
    collate_fn=data_collator,
)

tf_test_dataset = model.prepare_tf_dataset(
    tokenized_dataset["test"],
    shuffle=False,
    batch_size=32,
    collate_fn=data_collator,
)


In [56]:
from tf_keras.callbacks import EarlyStopping
import tensorflow as tf

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

In [57]:
history = model.fit(
    tf_train_dataset,
    validation_data=tf_val_dataset,
    epochs=5,
    callbacks=[early_stop]
)


Epoch 1/5
621/621 [==============================] - 377s 536ms/step - loss: 0.7238 - accuracy: 0.7864 - val_loss: 0.4723 - val_accuracy: 0.8461
Epoch 2/5
621/621 [==============================] - 314s 506ms/step - loss: 0.4107 - accuracy: 0.8651 - val_loss: 0.4375 - val_accuracy: 0.8510
Epoch 3/5
621/621 [==============================] - 315s 507ms/step - loss: 0.3275 - accuracy: 0.8931 - val_loss: 0.4468 - val_accuracy: 0.8510
Epoch 4/5
621/621 [==============================] - 313s 503ms/step - loss: 0.2599 - accuracy: 0.9178 - val_loss: 0.4924 - val_accuracy: 0.8315


In [58]:
model.compile(
    optimizer=Adam(learning_rate=2e-5),
    loss=SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [59]:
history = model.fit(
    tf_train_dataset,
    validation_data=tf_val_dataset,
    epochs=10,
    initial_epoch=5,
    callbacks=[early_stop]
)

Epoch 6/10
621/621 [==============================] - 356s 536ms/step - loss: 0.3480 - accuracy: 0.8842 - val_loss: 0.4442 - val_accuracy: 0.8453
Epoch 7/10
621/621 [==============================] - 315s 507ms/step - loss: 0.2505 - accuracy: 0.9189 - val_loss: 0.4979 - val_accuracy: 0.8408
Epoch 8/10
621/621 [==============================] - 313s 504ms/step - loss: 0.1645 - accuracy: 0.9510 - val_loss: 0.5602 - val_accuracy: 0.8469


In [60]:
model.compile(
    optimizer=Adam(learning_rate=1e-6),
    loss=SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [61]:
history = model.fit(
    tf_train_dataset,
    validation_data=tf_val_dataset,
    epochs=15,
    initial_epoch=10,
    callbacks=[early_stop]
)

Epoch 11/15
621/621 [==============================] - 356s 535ms/step - loss: 0.2285 - accuracy: 0.9291 - val_loss: 0.4451 - val_accuracy: 0.8501
Epoch 12/15
621/621 [==============================] - 315s 507ms/step - loss: 0.2102 - accuracy: 0.9368 - val_loss: 0.4555 - val_accuracy: 0.8469
Epoch 13/15
621/621 [==============================] - 313s 504ms/step - loss: 0.1995 - accuracy: 0.9402 - val_loss: 0.4658 - val_accuracy: 0.8485


In [62]:
loss, accuracy = model.evaluate(tf_train_dataset)
print(f"Train Accuracy: {accuracy*100:.2f}%")

loss, accuracy = model.evaluate(tf_val_dataset)
print(f"Validation Accuracy: {accuracy*100:.2f}%")

loss, accuracy = model.evaluate(tf_test_dataset)
print(f"Test Accuracy: {accuracy*100:.2f}%")


621/621 [==============================] - 112s 180ms/step - loss: 0.1931 - accuracy: 0.9435
Train Accuracy: 94.35%
78/78 [==============================] - 14s 178ms/step - loss: 0.4451 - accuracy: 0.8501
Validation Accuracy: 85.01%
78/78 [==============================] - 14s 179ms/step - loss: 0.4325 - accuracy: 0.8597
Test Accuracy: 85.97%


In [63]:
model.save_pretrained("/content/drive/MyDrive/emotion_model")
tokenizer.save_pretrained("/content/drive/MyDrive/emotion_model")

('/content/drive/MyDrive/emotion_model/tokenizer_config.json',
 '/content/drive/MyDrive/emotion_model/special_tokens_map.json',
 '/content/drive/MyDrive/emotion_model/vocab.txt',
 '/content/drive/MyDrive/emotion_model/added_tokens.json',
 '/content/drive/MyDrive/emotion_model/tokenizer.json')

In [64]:
labels = ['happy', 'sad', 'angry', 'fear', 'surprise', 'neutral']

def predict_emotion(text):
    inputs = tokenizer(
        text,
        return_tensors="tf",
        truncation=True,
        padding=True,
        max_length=128
    )
    outputs = model(inputs)
    predicted_class = int(tf.argmax(outputs.logits, axis=1))
    return labels[predicted_class]


In [65]:
print(predict_emotion("I am feeling very happy today"))
print(predict_emotion("This is the best day of my life"))
print(predict_emotion("I got selected for my dream job"))


TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


happy
happy
neutral


In [66]:
print(predict_emotion("I am feeling very sad and lonely"))
print(predict_emotion("I failed the exam and I feel hopeless"))
print(predict_emotion("Today is a very depressing day"))


sad
sad
sad


In [67]:
print(predict_emotion("I am extremely angry right now"))
print(predict_emotion("This situation makes me furious"))
print(predict_emotion("I hate how unfair this is"))


angry
angry
angry


In [68]:
print(predict_emotion("I am scared of failing the exam"))
print(predict_emotion("I feel very nervous about tomorrow"))
print(predict_emotion("I am afraid something bad will happen"))


fear
fear
fear


In [70]:
print(predict_emotion("I was surprised to see the results"))
print(predict_emotion("I did not expect this at all"))
print(predict_emotion("Wow, that was unexpected"))


surprise
surprise
surprise


In [71]:
print(predict_emotion("I am going to the office today"))
print(predict_emotion("It is a normal working day"))
print(predict_emotion("I have a meeting at 10 AM"))


neutral
neutral
neutral


In [72]:
print(predict_emotion("I am happy but also nervous about the future"))
print(predict_emotion("I was shocked and scared by the sudden noise"))
print(predict_emotion("Nothing special happened today"))


happy
surprise
neutral
